## Load Orders dataset

In [ ]:
import pandas as pd

file_path = '/Users/srujanpothina/Desktop/ecommerce-sales-analysis-global-superstore/Data/Raw/Global Superstore.xls'
orders = pd.read_excel(file_path, sheet_name="Orders")
orders.head()

## Convert Date Columns

Convert `Order Date` and `Ship Date` from object to datetime so they can be used for time-based analysis and date calculations.


In [ ]:
orders.dtypes

In [ ]:
orders['Order Date'] = pd.to_datetime(orders['Order Date'])
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'])

## Remove Duplicates

Remove duplicate rows to prevent inflated sales/profit metrics and ensure each order is counted once.


In [ ]:
before = len(orders)
orders.drop_duplicates(inplace=True)
after = len(orders)
print("Duplicates removed:", before - after)


## Handle Missing Values

Inspect and handle missing values, focusing on `Postal Code` while preserving valid transaction records.


In [ ]:
orders.isna().sum()

In [ ]:
# Example fills — adjust if needed
orders['Postal Code'] = orders['Postal Code'].fillna(0)

# If a column is irrelevant OR too missing, drop it:
# orders.drop(columns=['Postal Code'], inplace=True)

## Engineer New Columns

Create new time-based and performance features (Year, Month, Quarter, Ship Days, Profit per Item) to support richer analysis.

In [ ]:
orders['Year'] = orders['Order Date'].dt.year
orders['Month'] = orders['Order Date'].dt.month
orders['Quarter'] = orders['Order Date'].dt.quarter

In [ ]:
orders['Ship Days'] = (orders['Ship Date'] - orders['Order Date']).dt.days

## Quick Validation Checks

Run basic descriptive statistics and sanity checks to confirm that the cleaned data behaves as expected.

In [ ]:
orders['Ship Days'].describe()

In [ ]:
orders[['Sales','Profit','Quantity']].describe()

In [ ]:
orders.groupby('Year')['Sales'].sum()

## Save Cleaned Dataset

In [ ]:
orders.to_csv('/Users/srujanpothina/Desktop/ecommerce-sales-analysis-global-superstore/Data/Cleaned/orders_clean.csv', index=False)


## Load and Merge Returns Sheet


We use the Returns sheet to flag which orders were returned.  
This allows us to analyze return rates by region, product, and customer.

In [ ]:
returns = pd.read_excel(file_path, sheet_name="Returns")
returns.head()

In [ ]:
# Drop duplicates in returns if any
returns_before = len(returns)
returns.drop_duplicates(inplace=True)
returns_after = len(returns)
print("Returns duplicates removed:", returns_before - returns_after)

# Create Returned flag in orders
orders['Returned'] = orders['Order ID'].isin(returns['Order ID'])

# Quick check
orders['Returned'].value_counts()


## Load and Merge Peoples Sheet

The People sheet maps each Region to a sales contact/manager.  
We join this to the Orders data to enable manager-level performance analysis.

In [ ]:
people = pd.read_excel(file_path, sheet_name="People")
people.head()

In [ ]:
# Clean People sheet
people_before = len(people)
people.drop_duplicates(inplace=True)
people_after = len(people)
print("People duplicates removed:", people_before - people_after)

people.head()

In [ ]:
orders = orders.merge(people.rename(columns={'Person': 'Manager'}), 
                      how="left", on="Region")

orders[['Region', 'Manager']].head()


## 10. Save Final Master Cleaned Dataset

This dataset includes:
- Cleaned Orders data
- Returned flag
- Regional manager (from People sheet)
- Engineered features (Year, Month, Quarter, Ship Days, Profit per Item)

In [ ]:
orders.to_csv('/Users/srujanpothina/Desktop/ecommerce-sales-analysis-global-superstore/Data/Cleaned/orders_clean_master.csv', index=False)

In [ ]:
final = pd.read_csv("../Data/Cleaned/orders_clean_master.csv")
final.head()

## Data Cleaning Summary

### 1. Handling Missing Values

**Observation:**  
The dataset contains missing values in the `Postal Code` column.

**Decision:**  
Filled missing postal codes with `0`.

**Reasoning:**  
- `Postal Code` is not required for sales, profit, or forecasting.  
- Dropping rows would remove valid transactions.  
- Postal codes cannot be logically averaged or imputed.  
- Replacing with `0` preserves row count and clearly marks unknown values.

---

### 2. Removing Duplicates

**Observation:**  
The raw data may contain duplicate order records.

**Decision:**  
Removed duplicate rows from the Orders, Returns, and People sheets.

**Reasoning:**  
- Duplicate orders would artificially inflate sales and profit.  
- Duplicate returns or people records could lead to incorrect joins.  
- Ensures each order and mapping row is unique and reliable.

---

### 3. Converting Date Columns

**Decision:**  
Converted `Order Date` and `Ship Date` to datetime format.

**Reasoning:**  
- Enables time-based analysis (year, month, quarter).  
- Required for calculating shipping duration.  
- Prevents errors when grouping or sorting by date.

---

### 4. Feature Engineering

**Columns Created:**
- `Year`, `Month`, `Quarter`  
- `Ship Days` (days between Order Date and Ship Date)  
- `Profit per Item`  

**Reasoning:**  
- Time-based features support trend and seasonality analysis.  
- `Ship Days` allows evaluation of shipping performance.  
- `Profit per Item` highlights high and low margin products.

---

### 5. Merging Returns and People Sheets

**Decision:**  
- Added a `Returned` flag using the Returns sheet.  
- Joined the People sheet to map each Region to a manager/person.

**Reasoning:**  
- The `Returned` flag enables analysis of return rates by customer, product, and region.  
- Manager-level mapping supports performance insights by regional owner.

---

### 6. Saving Cleaned Datasets

**Decision:**  
Saved the cleaned Orders dataset and the enriched master dataset to:

- `../Data/Cleaned/orders_clean.csv`  
- `../Data/Cleaned/orders_clean_master.csv`  

**Reasoning:**  
- Separates raw and processed data (best practice).  
- Supports reusable analysis in SQL, dashboards, or additional notebooks.  
- Ensures reproducibility and a single source of truth for downstream work.


### Final Result

After cleaning and enrichment, the dataset is:

- Structured properly and consistently typed  
- Free of duplicate rows  
- Has no critical missing values in key numeric/date fields  
- Enriched with time-based and profitability features  
- Includes return flags and regional manager information  

Ready for **EDA, SQL exploration, and dashboard reporting.**
